In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import yfinance as yf
import talib
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
import torch.optim as optim
import os
from sklearn.model_selection import TimeSeriesSplit

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, log_loss, mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
import matplotlib.pyplot as plt

import optuna
from optuna.samplers import TPESampler
from optuna.trial import TrialState
from torch.optim.lr_scheduler import StepLR, ReduceLROnPlateau 
import shap
import plotly.graph_objs as go
import plotly.offline as pyo
from tqdm.auto import tqdm
from sklearn.utils.class_weight import compute_class_weight



In [2]:
if torch.cuda.is_available():
    device = torch.device('cuda')
    print("gpu")
else:
    device = torch.device('cpu')
print(torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('CUDA version:', torch.version.cuda)
print('cuDNN version:', torch.backends.cudnn.version())

gpu
2.1.2+cu121
CUDA available: True
CUDA version: 12.1
cuDNN version: 8902


In [3]:
start_date = '2018-01-01'
end_date = '2024-01-24'

stock_data = yf.download("AAPL", start=start_date, end=end_date)[["Adj Close", "High", "Low", "Volume"]]

stock_data = stock_data.reset_index()

stock_data = stock_data[['Date', 'Adj Close', "High", "Low", "Volume"]]

stock_data = stock_data.sort_values(by="Date")
stock_data.head(45)

[*********************100%%**********************]  1 of 1 completed


,Date,Adj Close,High,Low,Volume
0,2018-01-02,40.722870,43.075001,42.314999,102223600
1,2018-01-03,40.715790,43.637501,42.990002,118071600
2,2018-01-04,40.904903,43.367500,43.020000,89738400
3,2018-01-05,41.370632,43.842499,43.262501,94640000
4,2018-01-08,41.216965,43.902500,43.482498,82271200
5,2018-01-09,41.212242,43.764999,43.352501,86336000
6,2018-01-10,41.202770,43.575001,43.250000,95839600
7,2018-01-11,41.436817,43.872501,43.622501,74670800
8,2018-01-12,41.864700,44.340000,43.912498,101672400
9,2018-01-16,41.651939,44.847500,44.035000,118263600


In [4]:
time_step = 44

In [5]:
test_index = int((len(stock_data)-44)*0.8+44+44)

In [6]:
date = stock_data["Date"].iloc[time_step:].dt.strftime('%Y-%m-%d')
date_test = stock_data["Date"].iloc[test_index:].reset_index()
date_test.drop(columns=["index"], inplace=True)
date_test

,Date
0,2023-01-23
1,2023-01-24
2,2023-01-25
3,2023-01-26
4,2023-01-27
...,...
247,2024-01-17
248,2024-01-18
249,2024-01-19
250,2024-01-22


In [7]:
def add_technical_indicators(data, timeperiod=time_step):

    # MACD
    macd, macdsignal, macdhist = talib.MACD(data["Adj Close"], fastperiod=12, slowperiod=26, signalperiod=9)

    rsi = talib.RSI(data["Adj Close"], timeperiod=14)

    # CMO
    cmo = talib.CMO(data["Adj Close"], timeperiod=timeperiod)

    # MOM
    mom = talib.MOM(data["Adj Close"], timeperiod=timeperiod)

    # Bollinger Bands
    upperband, middleband, lowerband = talib.BBANDS(data["Adj Close"], timeperiod=20, nbdevup=2, nbdevdn=2, matype=0)

    # SMA
    sma_s = talib.SMA(data["Adj Close"], timeperiod=20)
    sma_l = talib.SMA(data["Adj Close"], timeperiod=50)

    # Calculate Exponential Moving Average (EMA)
    ema = talib.EMA(data["Adj Close"], timeperiod=timeperiod)

    # Calculate Stochastic Oscillator
    slowk, slowd = talib.STOCH(data['High'], data['Low'], data['Adj Close'], fastk_period=14, slowk_period=3, slowk_matype=0, slowd_period=3, slowd_matype=0)

    # Calculate Average True Range (ATR)
    atr = talib.ATR(data['High'], data['Low'], data['Adj Close'], timeperiod=14)

    # Calculate On-Balance Volume (OBV)
    obv = talib.OBV(data['Adj Close'], data['Volume'])

    # Combine all indicators into a DataFrame
    indicators = pd.DataFrame({
        'MACD': macd,
        'MACD_signal': macdsignal,
        'RSI': rsi,
        'CMO': cmo,
        'MOM': mom,
        'Upper_BB': upperband,
        'Middle_BB': middleband,
        'Lower_BB': lowerband,
        'SMA_SHORT': sma_s,
        'SMA_LONG': sma_l,
        'EMA': ema,
        'SLOWK': slowk,
        'SLOWD': slowd,
        'ATR': atr,
        'OBV': obv,

    })
    return indicators

In [8]:
indicators = add_technical_indicators(stock_data)
indicators.head(45)

,MACD,MACD_signal,RSI,CMO,MOM,Upper_BB,Middle_BB,Lower_BB,SMA_SHORT,SMA_LONG,EMA,SLOWK,SLOWD,ATR,OBV
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,102223600.0
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-15848000.0
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,73890400.0
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,168530400.0
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,86259200.0
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-76800.0
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-95916400.0
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-21245600.0
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,80426800.0
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-37836800.0


In [9]:
indicators_with_price = pd.concat([indicators, stock_data["Adj Close"]], axis=1, join='inner')
indicators_with_price.head(45)

,MACD,MACD_signal,RSI,CMO,MOM,Upper_BB,Middle_BB,Lower_BB,SMA_SHORT,SMA_LONG,EMA,SLOWK,SLOWD,ATR,OBV,Adj Close
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,102223600.0,40.722870
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-15848000.0,40.715790
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,73890400.0,40.904903
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,168530400.0,41.370632
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,86259200.0,41.216965
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-76800.0,41.212242
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-95916400.0,41.202770
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-21245600.0,41.436817
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,80426800.0,41.864700
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-37836800.0,41.651939


In [10]:
indicators_with_price = indicators_with_price.dropna()
indicators_with_price = indicators_with_price.reset_index(drop=True)
indicators_with_price

,MACD,MACD_signal,RSI,CMO,MOM,Upper_BB,Middle_BB,Lower_BB,SMA_SHORT,SMA_LONG,EMA,SLOWK,SLOWD,ATR,OBV,Adj Close
0,0.567013,0.443928,58.241572,7.100857,1.143597,43.382887,41.728833,40.074779,41.728833,40.814300,41.035722,-7.013413,-3.909660,2.715222,-3.124152e+08,42.355839
1,0.548493,0.464841,58.592059,7.332813,1.202900,43.261030,41.862708,40.464386,41.862708,40.847956,41.096608,-20.016882,-8.448448,2.714432,-2.214400e+08,42.405670
2,0.515804,0.475034,57.044886,6.516224,0.819328,43.280287,41.922405,40.564524,41.922405,40.878763,41.148143,-27.992068,-18.340788,2.690139,-3.790588e+08,42.256145
3,0.432811,0.466589,50.806400,3.052144,-0.254192,43.245417,41.956468,40.667519,41.956468,40.892875,41.168693,-36.985499,-28.331483,2.648797,-5.128460e+08,41.610508
4,0.361720,0.445615,50.674766,2.976561,-0.055672,43.183914,41.996702,40.809490,41.996702,40.897388,41.187696,-46.752042,-37.243203,2.644561,-5.914436e+08,41.596268
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1470,-1.918426,-1.197745,36.736038,-3.771153,-3.720001,199.131030,188.796999,178.462968,188.796999,189.292038,187.549134,26.310155,30.750307,3.101382,3.547580e+09,182.679993
1471,-1.555397,-1.269276,51.604604,4.598480,3.830002,198.242596,188.434000,178.625403,188.434000,189.536287,187.597173,33.195313,30.241850,3.341284,3.625586e+09,188.630005
1472,-1.019515,-1.219324,56.967975,8.324259,4.119995,197.297528,188.164999,179.032471,188.164999,189.787603,187.773298,51.916516,37.140661,3.339763,3.694327e+09,191.559998
1473,-0.402178,-1.055894,60.698088,11.147862,5.880005,197.121605,188.117999,179.114394,188.117999,190.033787,188.045152,76.309536,53.807122,3.370495,3.754460e+09,193.889999


In [11]:
# Define the conditions
conditions = [
    (indicators_with_price['RSI'] < 30), # al sinyali
    (indicators_with_price['RSI'] > 70) # sat sinyali
]

# Define the choices corresponding to each condition
choices = [2, 1] # 1 sat demek 2 al

# Use np.select to apply the conditions and choices
# The default value is 0 (for when the RSI is between 30 and 70)
indicators_with_price['RSI_SIGNAL'] = np.select(conditions, choices, default=0)

conditions = [
    (indicators_with_price['SMA_SHORT'] > indicators_with_price['SMA_LONG']), # al sinyali
    (indicators_with_price['SMA_LONG'] > indicators_with_price['SMA_SHORT']) # sat sinyali
]
# Define the choices corresponding to each condition
choices = [2, 1] # 1 sat demek 2 al

indicators_with_price['SMA_SIGNAL'] = np.select(conditions, choices, default=0)

# Display the dataframe to verify the results

# Create a column 'MACD_Signal' to hold the trading signal
indicators_with_price['MACD_Signal'] = 0

# Generate buy/sell signals
indicators_with_price['MACD_Signal'][indicators_with_price['MACD'] > indicators_with_price['MACD_signal']] = 2  # Buy signal
indicators_with_price['MACD_Signal'][indicators_with_price['MACD'] < indicators_with_price['MACD_signal']] = 1  # Sell signal

# Create a column 'BB_Signal' to hold the trading signal
indicators_with_price['BB_Signal'] = 0

# Generate buy/sell signals
indicators_with_price['BB_Signal'][indicators_with_price['Adj Close'] < indicators_with_price['Lower_BB']] = 2  # Buy signal
indicators_with_price['BB_Signal'][indicators_with_price['Adj Close'] > indicators_with_price['Upper_BB']] = 1  # Sell signal



indicators_with_price.head(50)

/tmp/ipykernel_43979/840148971.py:29: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipykernel_43979/840148971.py:30: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipykernel_43979/840148971.py:36: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipykernel_43979/840148971.py:37: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pyd

,MACD,MACD_signal,RSI,CMO,MOM,Upper_BB,Middle_BB,Lower_BB,SMA_SHORT,SMA_LONG,EMA,SLOWK,SLOWD,ATR,OBV,Adj Close,RSI_SIGNAL,SMA_SIGNAL,MACD_Signal,BB_Signal
0,0.567013,0.443928,58.241572,7.100857,1.143597,43.382887,41.728833,40.074779,41.728833,40.814300,41.035722,-7.013413,-3.909660,2.715222,-3.124152e+08,42.355839,0,2,2,0
1,0.548493,0.464841,58.592059,7.332813,1.202900,43.261030,41.862708,40.464386,41.862708,40.847956,41.096608,-20.016882,-8.448448,2.714432,-2.214400e+08,42.405670,0,2,2,0
2,0.515804,0.475034,57.044886,6.516224,0.819328,43.280287,41.922405,40.564524,41.922405,40.878763,41.148143,-27.992068,-18.340788,2.690139,-3.790588e+08,42.256145,0,2,2,0
3,0.432811,0.466589,50.806400,3.052144,-0.254192,43.245417,41.956468,40.667519,41.956468,40.892875,41.168693,-36.985499,-28.331483,2.648797,-5.128460e+08,41.610508,0,2,1,0
4,0.361720,0.445615,50.674766,2.976561,-0.055672,43.183914,41.996702,40.809490,41.996702,40.897388,41.187696,-46.752042,-37.243203,2.644561,-5.914436e+08,41.596268,0,2,1,0
5,0.226726,0.401838,42.776424,-1.895762,-1.685970,43.175299,41.999076,40.822853,41.999076,40.886127,41.163973,-59.960211,-47.899251,2.611109,-7.396632e+08,40.653912,0,2,1,2
6,0.072555,0.335981,38.805926,-4.708028,-2.298214,43.330934,41.955756,40.580579,41.955756,40.863472,41.115773,-60.364744,-55.692332,2.604322,-9.056264e+08,40.079487,0,2,1,2
7,-0.123098,0.244165,33.410026,-9.019855,-3.037197,43.669841,41.830426,39.991012,41.830426,40.822444,41.028467,-57.037842,-59.120932,2.589764,-1.069742e+09,39.151386,0,2,1,2
8,-0.126721,0.169988,48.771956,0.230814,-0.833458,43.603894,41.756843,39.909791,41.756843,40.813907,41.027645,-35.113187,-50.838591,2.699325,-9.195768e+08,41.009972,0,2,1,0
9,-0.212000,0.093590,42.761333,-4.462424,-1.894463,43.620645,41.637565,39.654485,41.637565,40.775782,40.980124,-25.755897,-39.302308,2.704910,-1.083267e+09,39.958424,0,2,1,0


In [12]:
# Sum the signals for each row
indicators_with_price['Sum_Buy_Signals'] = (indicators_with_price[['RSI_SIGNAL', 'SMA_SIGNAL', 'MACD_Signal', 'BB_Signal']] == 2).sum(axis=1)
indicators_with_price['Sum_Sell_Signals'] = (indicators_with_price[['RSI_SIGNAL', 'SMA_SIGNAL', 'MACD_Signal', 'BB_Signal']] == 1).sum(axis=1)

# Define the conditions
conditions = [
    (indicators_with_price['Sum_Buy_Signals'] >= 3), # At least three buy signals - Strong Buy
    (indicators_with_price['Sum_Buy_Signals'] >= 2), # At least two buy signals - Buy
    (indicators_with_price['Sum_Sell_Signals'] >= 3), # At least three sell signals - Strong Sell
    (indicators_with_price['Sum_Sell_Signals'] >= 2), # At least two sell signals - Sell
]

# Define the choices corresponding to each condition
choices = [4, 3, 1, 2] # Strong Buy, Buy, Strong Sell, Sell

# Use np.select to apply the conditions and choices
# The default value is 0 (Neutral)
indicators_with_price['Signal'] = np.select(conditions, choices, default=0)

# Remove the sum columns if they are no longer needed
indicators_with_price = indicators_with_price.drop(columns=['Sum_Buy_Signals', 'Sum_Sell_Signals'])

# Display the dataframe to verify the results
indicators_with_price[['RSI_SIGNAL', 'SMA_SIGNAL', 'MACD_Signal', 'BB_Signal', 'Signal']].head(50)

,RSI_SIGNAL,SMA_SIGNAL,MACD_Signal,BB_Signal,Signal
0,0,2,2,0,3
1,0,2,2,0,3
2,0,2,2,0,3
3,0,2,1,0,0
4,0,2,1,0,0
5,0,2,1,2,3
6,0,2,1,2,3
7,0,2,1,2,3
8,0,2,1,0,0
9,0,2,1,0,0


In [13]:
signal_distribution = indicators_with_price['Signal'].value_counts().sort_index()
signal_distribution

Signal
0    697
1      4
2    263
3    504
4      7
Name: count, dtype: int64

In [14]:
# indicators_with_price['Next_Adj_Close'] = indicators_with_price['Adj Close'].shift(-1)
# indicators_with_price['Return'] = ((indicators_with_price['Next_Adj_Close'] - indicators_with_price['Adj Close'])/indicators_with_price['Adj Close'])*100
# indicators_with_price['Signal'] = np.where(indicators_with_price['Return'] > 1, 1,
#                                            np.where(indicators_with_price['Return'] < -1, 2, 0))
# indicators_with_price


In [15]:
indicators_with_price["Signal"].value_counts()

Signal
0    697
3    504
2    263
4      7
1      4
Name: count, dtype: int64

In [16]:
# indicators_with_price = indicators_with_price.drop(columns=['Next_Adj_Close', 'Return'])
# indicators_with_price

In [17]:
indicators_with_price.drop(columns=['RSI_SIGNAL', 'SMA_SIGNAL', 'MACD_Signal', 'BB_Signal'], inplace=True)
indicators_with_price.head(50)

,MACD,MACD_signal,RSI,CMO,MOM,Upper_BB,Middle_BB,Lower_BB,SMA_SHORT,SMA_LONG,EMA,SLOWK,SLOWD,ATR,OBV,Adj Close,Signal
0,0.567013,0.443928,58.241572,7.100857,1.143597,43.382887,41.728833,40.074779,41.728833,40.814300,41.035722,-7.013413,-3.909660,2.715222,-3.124152e+08,42.355839,3
1,0.548493,0.464841,58.592059,7.332813,1.202900,43.261030,41.862708,40.464386,41.862708,40.847956,41.096608,-20.016882,-8.448448,2.714432,-2.214400e+08,42.405670,3
2,0.515804,0.475034,57.044886,6.516224,0.819328,43.280287,41.922405,40.564524,41.922405,40.878763,41.148143,-27.992068,-18.340788,2.690139,-3.790588e+08,42.256145,3
3,0.432811,0.466589,50.806400,3.052144,-0.254192,43.245417,41.956468,40.667519,41.956468,40.892875,41.168693,-36.985499,-28.331483,2.648797,-5.128460e+08,41.610508,0
4,0.361720,0.445615,50.674766,2.976561,-0.055672,43.183914,41.996702,40.809490,41.996702,40.897388,41.187696,-46.752042,-37.243203,2.644561,-5.914436e+08,41.596268,0
5,0.226726,0.401838,42.776424,-1.895762,-1.685970,43.175299,41.999076,40.822853,41.999076,40.886127,41.163973,-59.960211,-47.899251,2.611109,-7.396632e+08,40.653912,3
6,0.072555,0.335981,38.805926,-4.708028,-2.298214,43.330934,41.955756,40.580579,41.955756,40.863472,41.115773,-60.364744,-55.692332,2.604322,-9.056264e+08,40.079487,3
7,-0.123098,0.244165,33.410026,-9.019855,-3.037197,43.669841,41.830426,39.991012,41.830426,40.822444,41.028467,-57.037842,-59.120932,2.589764,-1.069742e+09,39.151386,3
8,-0.126721,0.169988,48.771956,0.230814,-0.833458,43.603894,41.756843,39.909791,41.756843,40.813907,41.027645,-35.113187,-50.838591,2.699325,-9.195768e+08,41.009972,0
9,-0.212000,0.093590,42.761333,-4.462424,-1.894463,43.620645,41.637565,39.654485,41.637565,40.775782,40.980124,-25.755897,-39.302308,2.704910,-1.083267e+09,39.958424,0


In [18]:
# y = indicators_with_price["Adj Close"]
# y_2 = indicators_with_price["SMA"]
# y_3 = indicators_with_price["EMA"]
# y_4 = indicators_with_price["Upper_BB"]
# y_5 = indicators_with_price["Middle_BB"]
# y_6 = indicators_with_price["Lower_BB"]
# X = np.array(date)

# trace = go.Scatter(x=X, y=y, mode="lines", name="Adj Close")
# trace_2 = go.Scatter(x=X, y=y_2, mode="lines", name="SMA")
# trace_3 = go.Scatter(x=X, y=y_3, mode="lines", name="EMA")
# trace_4 = go.Scatter(x=X, y=y_4, mode="lines", name="Upper_BB")
# trace_5 = go.Scatter(x=X, y=y_5, mode="lines", name="Middle_BB")
# trace_6 = go.Scatter(x=X, y=y_6, mode="lines", name="Lower_BB")



# layout = go.Layout(
#     title='Stock Price and Volume',
#     xaxis=dict(title='Date'),
#     yaxis=dict(title='Adj Close', side='left', rangemode='tozero'),
#     yaxis2=dict(title='SMA', side='right', overlaying='y', rangemode='tozero'),
#     yaxis3=dict(title='EMA', side='right', overlaying='y', rangemode='tozero'),
#     yaxis4=dict(title='Upper_BB', side='right', overlaying='y', rangemode='tozero'),
#     yaxis5=dict(title='Middle_BB', side='right', overlaying='y', rangemode='tozero'),
#     yaxis6=dict(title='Lower_BB', side='right', overlaying='y', rangemode='tozero'),
#     height=600,
# )

# fig = go.Figure(data=[trace, trace_2, trace_3, trace_4, trace_5, trace_6], layout=layout)

# # Show plot
# pyo.iplot(fig)

In [19]:
class RollingWindowDataset(Dataset):
    def __init__(self, X, y, window_size, device="gpu"):
        self.X = X.clone().detach().to(torch.float).to(device)
        self.y = y.clone().detach().to(torch.float).to(device)
        self.window_size = window_size

    def __len__(self):
        # Adjust the length to account for window size
        return len(self.X) - self.window_size 

    def __getitem__(self, idx):
        # Ensure idx is within the valid range
        if idx + self.window_size > len(self.X):
            raise IndexError("Index out of bounds")

        # Extract a window of data from X
        X_window = self.X[idx : idx + self.window_size]
        
        # Assuming you're predicting the value right after the window
        y_target = self.y[idx + self.window_size]  

        return X_window.clone().detach().to(torch.float), y_target.clone().detach().to(torch.float)


In [20]:
X = indicators_with_price.iloc[:, :-1]
y = indicators_with_price.iloc[:,-1]

signal_true = indicators_with_price.iloc[:,-1]
y

0       3
1       3
2       3
3       0
4       0
       ..
1470    2
1471    2
1472    0
1473    0
1474    0
Name: Signal, Length: 1475, dtype: int64

In [21]:
train_signal_true = signal_true.iloc[:int(len(X)*0.8)]
test_signal_true = signal_true.iloc[int(len(X)*0.8):]
test_signal_true

1180    0
1181    0
1182    3
1183    3
1184    3
       ..
1470    2
1471    2
1472    0
1473    0
1474    0
Name: Signal, Length: 295, dtype: int64

In [22]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
# X_train = X_train.to_numpy()
# y_train = y_train.to_numpy()
y_test.head(44)

1180    0
1181    0
1182    3
1183    3
1184    3
1185    3
1186    3
1187    2
1188    2
1189    0
1190    0
1191    0
1192    0
1193    0
1194    0
1195    0
1196    0
1197    0
1198    0
1199    3
1200    3
1201    2
1202    2
1203    2
1204    2
1205    2
1206    2
1207    2
1208    2
1209    2
1210    2
1211    2
1212    2
1213    2
1214    0
1215    0
1216    0
1217    0
1218    0
1219    0
1220    0
1221    0
1222    0
1223    2
Name: Signal, dtype: int64

In [23]:
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
class_weights

array([ 0.42369838, 59.        ,  1.15686275,  0.57420925, 59.        ])

In [24]:
# # Get the count of each unique value in the series
# category_counts = y_train.value_counts()
# print(category_counts.index)
# print(category_counts.values)

# # Create bar plot
# plt.bar(["1","0"], category_counts.values)

# # Adding labels and title
# plt.xlabel('Category')
# plt.ylabel('Frequency')
# plt.title('Distribution of Categories')

# # Show plot
# plt.show()

In [25]:
X_train_df = pd.DataFrame()
X_test_df = pd.DataFrame()
scaler_dict = {}

for column in X_train.columns:
    # Initialize a scaler
    scaler = MinMaxScaler()
    
    # Fit on the training data and transform it
    X_train_scaled = scaler.fit_transform(X_train[column].to_numpy().reshape(-1, 1))
    X_train_df[column] = X_train_scaled.flatten()
    
    # Transform the test data using the same scaler
    X_test_scaled = scaler.transform(X_test[column].to_numpy().reshape(-1, 1))
    X_test_df[column] = X_test_scaled.flatten()

    scaler_dict[column] = scaler

    

X_train_df.head(24)

features = X_train_df.columns
X_test_df

,MACD,MACD_signal,RSI,CMO,MOM,Upper_BB,Middle_BB,Lower_BB,SMA_SHORT,SMA_LONG,EMA,SLOWK,SLOWD,ATR,OBV,Adj Close
0,0.499847,0.411144,0.517394,0.387563,0.487569,0.813419,0.791684,0.751875,0.791684,0.814103,0.833706,0.876576,0.853920,0.764804,0.775662,0.793797
1,0.523637,0.432238,0.527281,0.393263,0.448749,0.816205,0.793223,0.752021,0.793223,0.813227,0.835055,0.895160,0.867699,0.724522,0.788577,0.797684
2,0.523008,0.448973,0.456307,0.359879,0.379729,0.815520,0.792793,0.751879,0.792793,0.810946,0.835217,0.907286,0.887937,0.685712,0.778442,0.775317
3,0.534244,0.464868,0.497611,0.382097,0.444493,0.813808,0.792104,0.752318,0.792104,0.810434,0.836117,0.913913,0.901248,0.661728,0.787383,0.790115
4,0.547370,0.480511,0.514479,0.391191,0.467133,0.815320,0.792814,0.752139,0.792814,0.809834,0.837282,0.923475,0.911343,0.623616,0.797445,0.796184
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
290,0.331987,0.360467,0.219194,0.330142,0.438975,1.105702,1.104494,1.078159,1.104494,1.140327,1.152418,0.696649,0.699736,0.263755,0.912203,1.018693
291,0.358007,0.354748,0.456177,0.434229,0.530971,1.099583,1.101858,1.079371,1.101858,1.142176,1.152792,0.728273,0.697238,0.316961,0.925666,1.059493
292,0.396416,0.358741,0.541662,0.480564,0.534505,1.093074,1.099905,1.082407,1.099905,1.144078,1.154161,0.814261,0.731129,0.316623,0.937531,1.079584
293,0.440664,0.371807,0.601114,0.515679,0.555951,1.091863,1.099564,1.083019,1.099564,1.145941,1.156274,0.926300,0.813002,0.323439,0.947909,1.095561


In [26]:
scaler_dict

{'MACD': MinMaxScaler(),
 'MACD_signal': MinMaxScaler(),
 'RSI': MinMaxScaler(),
 'CMO': MinMaxScaler(),
 'MOM': MinMaxScaler(),
 'Upper_BB': MinMaxScaler(),
 'Middle_BB': MinMaxScaler(),
 'Lower_BB': MinMaxScaler(),
 'SMA_SHORT': MinMaxScaler(),
 'SMA_LONG': MinMaxScaler(),
 'EMA': MinMaxScaler(),
 'SLOWK': MinMaxScaler(),
 'SLOWD': MinMaxScaler(),
 'ATR': MinMaxScaler(),
 'OBV': MinMaxScaler(),
 'Adj Close': MinMaxScaler()}

In [27]:
correlation_matrix = X_train_df.corr()

# Create the heatmap with text
fig = go.Figure(data=go.Heatmap(
                    z=correlation_matrix,
                    x=correlation_matrix.columns,
                    y=correlation_matrix.columns,
                    colorscale='Viridis',
                    text=correlation_matrix.round(2).values,  # Rounded values for display
                    texttemplate="%{text}",
                    textfont={"size":9}  # Adjust text size if necessary
                    ))

# Update the layout
fig.update_layout(
    title='Correlation Matrix',
    xaxis_title="Variables",
    yaxis_title="Variables",
    xaxis=dict(side='bottom'),
    yaxis=dict(autorange='reversed'),
    width=1000,  # or any width you desire
    height=1000,  # or any height you desire
)

# Show the figure
pyo.iplot(fig)

In [28]:
X_train_tensor = torch.tensor(X_train_df.to_numpy(), dtype=torch.float, device=device)
y_train_tensor = torch.tensor(y_train.to_numpy(), dtype=torch.float, device=device)

X_test_tensor = torch.tensor(X_test_df.to_numpy(), dtype=torch.float, device=device)
y_test_tensor = torch.tensor(y_test.to_numpy(), dtype=torch.float, device=device)

train_data = RollingWindowDataset(X_train_tensor, y_train_tensor, window_size=time_step, device=device)
test_data = RollingWindowDataset(X_test_tensor, y_test_tensor, window_size=time_step, device=device)

print(test_data.__getitem__(0)[1])
print(test_data.__getitem__(0)[0])


tensor(2., device='cuda:0')
tensor([[0.4998, 0.4111, 0.5174, 0.3876, 0.4876, 0.8134, 0.7917, 0.7519, 0.7917,
         0.8141, 0.8337, 0.8766, 0.8539, 0.7648, 0.7757, 0.7938],
        [0.5236, 0.4322, 0.5273, 0.3933, 0.4487, 0.8162, 0.7932, 0.7520, 0.7932,
         0.8132, 0.8351, 0.8952, 0.8677, 0.7245, 0.7886, 0.7977],
        [0.5230, 0.4490, 0.4563, 0.3599, 0.3797, 0.8155, 0.7928, 0.7519, 0.7928,
         0.8109, 0.8352, 0.9073, 0.8879, 0.6857, 0.7784, 0.7753],
        [0.5342, 0.4649, 0.4976, 0.3821, 0.4445, 0.8138, 0.7921, 0.7523, 0.7921,
         0.8104, 0.8361, 0.9139, 0.9012, 0.6617, 0.7874, 0.7901],
        [0.5474, 0.4805, 0.5145, 0.3912, 0.4671, 0.8153, 0.7928, 0.7521, 0.7928,
         0.8098, 0.8373, 0.9235, 0.9113, 0.6236, 0.7974, 0.7962],
        [0.5399, 0.4914, 0.4471, 0.3605, 0.4592, 0.8163, 0.7941, 0.7537, 0.7941,
         0.8092, 0.8374, 0.9237, 0.9172, 0.5981, 0.7914, 0.7760],
        [0.5112, 0.4936, 0.3677, 0.3216, 0.4080, 0.8078, 0.7900, 0.7546, 0.7900,
         

# Vanilla LSTM

In [29]:
class VanillaLSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_size, layer_size, output_dim, dropout_prob):
        super(VanillaLSTMModel, self).__init__()

        self.hidden_size = hidden_size
        self.layer_size = layer_size

        self.lstm = nn.LSTM(input_size = input_dim, hidden_size = hidden_size, num_layers=layer_size,
                            dropout=(dropout_prob if layer_size > 1 else 0), batch_first=True)
                            
        self.dropout = nn.Dropout(dropout_prob)
        
        self.fc = nn.Linear(hidden_size, output_dim)

    def forward(self, x):
        # Initializing hidden state
        h0 = torch.zeros(self.layer_size, x.size(0), self.hidden_size).to(device) 

        # Initialize cell state
        c0 = torch.zeros(self.layer_size, x.size(0), self.hidden_size).to(device) 

        # Forward propagate LSTM
        out, (hn, cn) = self.lstm(x, (h0.detach(), c0.detach()))

        out = self.dropout(out[:, -1, :])  # Add dropout

        out = self.fc(out)

        return out

# Stacked LSTM

In [30]:
class StackedLSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_size, layer_size, output_dim, dropout_prob, stateful):
        super(StackedLSTMModel, self).__init__()

        self.hidden_size = hidden_size
        self.layer_size = layer_size
        self.stateful = stateful
        
        self.hn = None
        self.cn = None

        self.lstm = nn.LSTM(input_size = input_dim, hidden_size = hidden_size, num_layers=self.layer_size,
                            dropout=(dropout_prob if self.layer_size > 1 else 0), batch_first=True)
                            
        self.dropout = nn.Dropout(dropout_prob)
        
        self.fc = nn.Linear(hidden_size, output_dim)

    def reset_hidden_state(self, batch_size=None):
        if batch_size is None or not self.stateful:
            # Reset hidden and cell states to None if batch_size is not provided or if the model is not stateful
            self.hn = None
            self.cn = None
        else:
            # Resize hidden and cell states to match current batch size, preserving the state as much as possible
            self.hn = self._resize_state(self.hn, batch_size)
            self.cn = self._resize_state(self.cn, batch_size)


    def _resize_state(self, state, batch_size):
        if state is None:
            # If state is not initialized, initialize with zeros
            return torch.zeros(self.layer_size, batch_size, self.hidden_size).to(device)
        elif batch_size < state.size(1):
            # If batch size is smaller, truncate the state
            return state[:, :batch_size, :].contiguous()
        elif batch_size > state.size(1):
            # If batch size is larger, pad the state with zeros
            padding_size = batch_size - state.size(1)
            padding = torch.zeros(self.layer_size, padding_size, self.hidden_size).to(device)
            return torch.cat([state, padding], dim=1)
        

    def forward(self, x):
        current_batch_size = x.size(0)

        # Check and adjust the hidden and cell states
        if not self.stateful or self.hn is None or self.hn.size(1) != current_batch_size:
            self.reset_hidden_state(current_batch_size)
        else:
            # Detach the hidden states from the graph to avoid backpropagating through the entire sequence history
            self.hn = self.hn.detach()
            self.cn = self.cn.detach()

        # Ensure that hn and cn are not None and have the correct shape
        h0 = self.hn if self.hn is not None else torch.zeros(self.layer_size, current_batch_size, self.hidden_size).to(device)
        c0 = self.cn if self.cn is not None else torch.zeros(self.layer_size, current_batch_size, self.hidden_size).to(device)

        # Forward propagate LSTM
        out, (hn, cn) = self.lstm(x, (h0, c0))

        # Update hidden states if stateful
        if self.stateful:
            self.hn, self.cn = hn.detach(), cn.detach()

        # Process the output of the last time step
        out = self.dropout(out[:, -1, :])  # Add dropout
        out = self.fc(out)  # Fully connected layer

        return out

# Bidirectional LSTM

# 1D CNN-LSTM

In [31]:
class OneDimCNNLSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_size, layer_size, output_dim, dropout_prob, conv_channels, kernel_size, pool_size, stride):
        super(OneDimCNNLSTMModel, self).__init__()

        self.hidden_size = hidden_size
        self.layer_size = layer_size

        # Convolutional Layer
        self.conv1 = nn.Conv1d(in_channels=input_dim, out_channels=conv_channels, kernel_size=kernel_size)
        self.relu1 = nn.ReLU()
        self.maxpool1 = nn.MaxPool1d(kernel_size=pool_size, stride=pool_size)

        # self.conv2 = nn.Conv1d(in_channels=conv_channels, out_channels=conv_channels*2, kernel_size=kernel_size)
        # self.relu2 = nn.ReLU()
        # self.maxpool2 = nn.MaxPool1d(kernel_size=2, stride=2)


        self.lstm = nn.LSTM(input_size = conv_channels, hidden_size = hidden_size, num_layers=layer_size,
                            dropout=(dropout_prob if layer_size > 1 else 0), batch_first=True)
                            
        self.dropout = nn.Dropout(dropout_prob)
        
        self.fc = nn.Linear(hidden_size, output_dim)

    def forward(self, x):
        # CNN in_dim: (batch_size, in_channels, length)
        x = x.transpose(1, 2)

        # Conv layer forward propagate
        x = self.conv1(x)
        x = self.relu1(x)
        x = self.maxpool1(x)

        # x = self.conv2(x)
        # x = self.relu2(x)
        # x = self.maxpool2(x)

        # LSTM in_dim: (batch_size, seq_len, input_size)
        x = x.transpose(1, 2)

        assert x.size(-1) == self.lstm.input_size, f"Mismatch in LSTM input size. Expected: {self.lstm.input_size}, Got: {x.size(-1)}"

        # Initializing hidden state
        h0 = torch.zeros(self.layer_size, x.size(0), self.hidden_size).to(device) 

        # Initialize cell state
        c0 = torch.zeros(self.layer_size, x.size(0), self.hidden_size).to(device) 

        # Forward propagate LSTM
        out, (hn, cn) = self.lstm(x, (h0.detach(), c0.detach()))

        out = self.dropout(out[:, -1, :])  # Add dropout

        out = self.fc(out)

        return out

In [32]:
class ModelActioner:
    
    def __init__(self, train_data, test_data, device, model_type):
        self.train_data = train_data
        self.test_data = test_data
        self.device = device
        self.model_type = model_type
        self.model = None
        self.optimizer = None
        self.criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

    
    def train_validate(self, config, trial):

        if self.model_type == "Vanilla LSTM":
            batch_size = config["batch_size"]
            epochs = config["epochs"]
            hidden_size = config["hidden_size"]
            num_layers = 1
            learning_rate = config["learning_rate"]
            dropout_prob = config["dropout_prob"]
            weight_decay = config["weight_decay"]
            lr_step_size = config["lr_step_size"]
            gamma = config["gamma"]

            self.model = VanillaLSTMModel(input_dim=self.train_data.__getitem__(0)[0].shape[1], hidden_size=hidden_size, layer_size=num_layers, dropout_prob=dropout_prob, output_dim=5).to(self.device)
            
        elif self.model_type == "Stacked LSTM":
            batch_size = config["batch_size"]
            epochs = config["epochs"]
            hidden_size = config["hidden_size"]
            num_layers = config["num_layers"]
            learning_rate = config["learning_rate"]
            dropout_prob = config["dropout_prob"]
            weight_decay = config["weight_decay"]
            lr_step_size = config["lr_step_size"]
            gamma = config["gamma"]
            
            self.model = StackedLSTMModel(input_dim=self.train_data.__getitem__(0)[0].shape[1], hidden_size=hidden_size, layer_size=num_layers, dropout_prob=dropout_prob, stateful=False, output_dim=5).to(self.device)

        elif self.model_type == "1D CNN-LSTM":
            batch_size = config["batch_size"]
            epochs = config["epochs"]
            hidden_size = config["hidden_size"]
            num_layers = config["num_layers"]
            learning_rate = config["learning_rate"]
            dropout_prob = config["dropout_prob"]
            weight_decay = config["weight_decay"]
            lr_step_size = config["lr_step_size"]
            gamma = config["gamma"]
            kernel_size = config["kernel_size"]
            conv_channels = config["conv_channels"]
            pool_size = config["pool_size"]
            stride = config["stride"]

            self.model = OneDimCNNLSTMModel(input_dim=self.train_data.__getitem__(0)[0].shape[1], hidden_size=hidden_size, layer_size=num_layers, dropout_prob=dropout_prob, output_dim=5, conv_channels=conv_channels, kernel_size=kernel_size, pool_size=pool_size, stride=stride).to(self.device)

        n_splits = 5
        tscv = TimeSeriesSplit(n_splits=n_splits)

        val_losses = []

        for fold, (train_ids, val_ids) in enumerate(tscv.split(self.train_data)):
            print(f'Starting fold {fold+1}:')

            self.optimizer = optim.Adam(self.model.parameters(), lr = learning_rate, weight_decay=weight_decay)

            scheduler = ReduceLROnPlateau(self.optimizer, patience=lr_step_size, factor=gamma, mode="min", verbose=True) 

            train_subset = Subset(self.train_data, train_ids)
            val_subset = Subset(self.train_data, val_ids)
            
            # Creating data loader
            train_loader = DataLoader(dataset=train_subset, batch_size=batch_size, shuffle=False)
            val_loader = DataLoader(dataset=val_subset, batch_size=batch_size, shuffle=False)

            # Training Loop
            for epoch in range(epochs):
                print('epochs {}/{}'.format(epoch+1,epochs))

                running_loss = 0.0
                total_sample_train = 0

                self.model.train()

                for batch_idx, (data, target) in enumerate(train_loader):
                    data, target = data.to(self.device), target.to(self.device)
                    target = target.long()


                    self.optimizer.zero_grad()
                    preds = self.model(data)

                    loss = self.criterion(preds, target)
                    loss = loss.mean()
                    loss.backward()
                    self.optimizer.step() # Update model params

                    running_loss += loss.item() * data.size(0)
                    total_sample_train += data.size(0)

                train_loss = running_loss/total_sample_train
                #print(f"train loss: {train_loss}")

                self.model.eval()
                val_running_loss = 0.0
                total_sample_val = 0

                with torch.no_grad():

                    for batch_idx, (data, target) in enumerate(val_loader):
                        data, target = data.to(self.device), target.to(self.device)
                        target = target.long()

                        preds = self.model(data)
                        loss = self.criterion(preds, target)
                        loss = loss.mean()

                        val_running_loss += loss.item() * data.size(0)
                        total_sample_val += data.size(0)
                
                if trial.should_prune():
                    raise optuna.TrialPruned()
                
                val_loss = val_running_loss/total_sample_val
                val_losses.append(val_loss)
                scheduler.step(val_loss)
                
                unique_step = fold * epochs + epoch
                trial.report(val_loss, unique_step)

                current_lr = self.optimizer.param_groups[0]['lr']

                print(f'Current Learning Rate: {current_lr}')
                print(f"train_loss: {train_loss}, val_loss: {val_loss}")
                
        mean_val_loss = np.mean(val_losses)
        print(f"Mean validation loss: {mean_val_loss}")
        return mean_val_loss
    
                    
    def train(self, config):
        if self.model_type == "Vanilla LSTM":

            batch_size = config["batch_size"]
            epochs = config["epochs"]
            hidden_size = config["hidden_size"]
            num_layers = 1
            learning_rate = config["learning_rate"]
            dropout_prob = config["dropout_prob"]
            weight_decay = config["weight_decay"]
            lr_step_size = config["lr_step_size"]
            gamma = config["gamma"]

            self.model = VanillaLSTMModel(input_dim=self.train_data.__getitem__(0)[0].shape[1], hidden_size=hidden_size, layer_size=num_layers, dropout_prob=dropout_prob, output_dim=5).to(self.device)
            
        elif self.model_type == "Stacked LSTM":
            
            batch_size = config["batch_size"]
            epochs = config["epochs"]
            hidden_size = config["hidden_size"]
            num_layers = config["num_layers"]
            learning_rate = config["learning_rate"]
            dropout_prob = config["dropout_prob"]
            weight_decay = config["weight_decay"]
            lr_step_size = config["lr_step_size"]
            gamma = config["gamma"]
            
            self.model = StackedLSTMModel(input_dim=self.train_data.__getitem__(0)[0].shape[1], hidden_size=hidden_size, layer_size=num_layers, dropout_prob=dropout_prob, stateful=False, output_dim=5).to(self.device)

        elif self.model_type == "1D CNN-LSTM":
            batch_size = config["batch_size"]
            epochs = config["epochs"]
            hidden_size = config["hidden_size"]
            num_layers = config["num_layers"]
            learning_rate = config["learning_rate"]
            dropout_prob = config["dropout_prob"]
            weight_decay = config["weight_decay"]
            lr_step_size = config["lr_step_size"]
            gamma = config["gamma"]
            kernel_size = config["kernel_size"]
            conv_channels = config["conv_channels"]
            pool_size = config["pool_size"]
            stride = config["stride"]

            self.model = OneDimCNNLSTMModel(input_dim=self.train_data.__getitem__(0)[0].shape[1], hidden_size=hidden_size, layer_size=num_layers, dropout_prob=dropout_prob, output_dim=5, conv_channels=conv_channels, kernel_size=kernel_size, pool_size=pool_size, stride=stride).to(self.device)

        # Update optimizer with updated lr
        self.optimizer = optim.Adam(self.model.parameters(), lr = learning_rate, weight_decay=weight_decay)

        # Creating data loader
        train_loader = DataLoader(dataset=self.train_data, batch_size=batch_size, shuffle=False)

        scheduler = ReduceLROnPlateau(self.optimizer, patience=lr_step_size, factor=gamma, mode="min", verbose=True)  

        # Training Loop
        for epoch in range(epochs):
            print('epochs {}/{}'.format(epoch+1,epochs))

            running_loss = 0.0
            total_sample_train = 0

            self.model.train()

            for batch_idx, (data, target) in enumerate(train_loader):

                data, target = data.to(self.device), target.to(self.device)
                target = target.long()  

                self.optimizer.zero_grad()
                preds = self.model(data)

                loss = self.criterion(preds, target)
                loss = loss.mean()
                loss.backward()
                self.optimizer.step() # Update model params

                running_loss += loss.item() * data.size(0)
                total_sample_train += data.size(0)

            train_loss = running_loss/total_sample_train
            #print(f"train loss: {train_loss}")
            scheduler.step(train_loss)
            current_lr = self.optimizer.param_groups[0]['lr']

            print(f'Current Learning Rate: {current_lr}')
            print(f"train_loss: {train_loss}")
        
        return self.model
            
    
    def test(self, config):
        batch_size = config["batch_size"]
        all_preds = []

        test_loader = DataLoader(dataset=self.test_data, batch_size=batch_size, shuffle=False)

        running_loss = .0
        total_sample = 0

        self.model.eval()

        with torch.no_grad():

            for batch_idx, (data, target) in enumerate(test_loader):

                data, target = data.to(self.device), target.to(self.device)
                target = target.long()
                
                preds = self.model(data)
                loss = self.criterion(preds, target)

                running_loss += loss.item() * data.size(0)
                total_sample += data.size(0)

                _, predicted_classes = torch.max(preds, 1)
                all_preds.extend(predicted_classes.cpu().numpy())

            test_loss = running_loss/total_sample
            print(f"test_loss: {test_loss}")

        return all_preds
    


In [33]:
model_type = "1D CNN-LSTM"

def objective(trial):
    if model_type == "Vanilla LSTM":
        config = {
            "batch_size": trial.suggest_int("batch_size", 32, 128, log=True),
            "epochs": trial.suggest_int("epochs", 50, 150),
            "hidden_size": trial.suggest_int("hidden_size", 10, 100),
            "learning_rate": trial.suggest_float("learning_rate", 1e-6, 1e-1, log=True),
            "dropout_prob": trial.suggest_float("dropout_prob", 0.0, 0.15),
            "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True),
            "lr_step_size": trial.suggest_int("lr_step_size", 10, 100), 
            "gamma": trial.suggest_float("gamma", 0.1, 0.9)
        }

    elif model_type == "Stacked LSTM":
        config = {
            "batch_size": trial.suggest_int("batch_size", 32, 128, log=True),
            "epochs": trial.suggest_int("epochs", 50, 150),
            "hidden_size": trial.suggest_int("hidden_size", 10, 100),
            "learning_rate": trial.suggest_float("learning_rate", 1e-6, 1e-1, log=True),
            "dropout_prob": trial.suggest_float("dropout_prob", 0.0, 0.2),
            "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True),
            "lr_step_size": trial.suggest_int("lr_step_size", 5, 100), 
            "gamma": trial.suggest_float("gamma", 0.1, 0.9),
            "num_layers": trial.suggest_int("num_layers", 1, 5)
        }

    elif model_type == "1D CNN-LSTM":
        config = {
            "batch_size": trial.suggest_int("batch_size", 32, 128, log=True),
            "epochs": trial.suggest_int("epochs", 50, 150),
            "hidden_size": trial.suggest_int("hidden_size", 10, 100),
            "learning_rate": trial.suggest_float("learning_rate", 1e-6, 1e-1, log=True),
            "dropout_prob": trial.suggest_float("dropout_prob", 0.0, 0.2),
            "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True),
            "lr_step_size": trial.suggest_int("lr_step_size", 5, 100), 
            "gamma": trial.suggest_float("gamma", 0.1, 0.9),
            "conv_channels": trial.suggest_int("conv_channels", 16, 128),
            "kernel_size": trial.suggest_int("kernel_size", 2, 6),
            "num_layers": trial.suggest_int("num_layers", 1, 5),
            "pool_size": trial.suggest_int("pool_size", 2, 5),
            "stride": trial.suggest_int("stride", 1, 4)
        }

    trainer = ModelActioner(train_data, test_data, device, model_type)

    val_loss = trainer.train_validate(config, trial)

    return val_loss

In [34]:
study_name = "Vanilla-LSTM-Tunner"
storage_url = "sqlite:///db.sqlite3"

optuna.delete_study(study_name=study_name, storage= storage_url)

study = optuna.create_study(direction='minimize', 
                            storage=storage_url, 
                            sampler=TPESampler(),
                            pruner=optuna.pruners.HyperbandPruner(
                            min_resource=30,  
                            max_resource=150,  
                            reduction_factor=3,  
                            ),
                            study_name=study_name,
                            load_if_exists=False)

pbar = tqdm(total=25, desc='Optimizing', unit='trial')

def callback(study, trial):
    # Update the progress bar
    pbar.update(1)
    # Optionally, you can include additional information in the progress bar
    pbar.set_postfix_str(f"Best Value: {study.best_value:.4f}")


study.optimize(objective, n_trials=25, callbacks=[callback])
pbar.close()

# Best hyperparameters
print('Number of finished trials:', len(study.trials))
print('Best trial:')
trial = study.best_trial

print('Value:', trial.value)
print('Params:')
for key, value in trial.params.items():
    print(f'{key}: {value}')

[I 2024-01-27 01:50:51,617] A new study created in RDB with name: Vanilla-LSTM-Tunner


Optimizing:   0%|          | 0/25 [00:00<?, ?trial/s]

Starting fold 1:
epochs 1/84
Current Learning Rate: 0.04471910604647985
train_loss: 1.570634452460324, val_loss: 1.5423387509805184
epochs 2/84
Current Learning Rate: 0.04471910604647985
train_loss: 1.2768523849117819, val_loss: 1.3679035946174904
epochs 3/84
Current Learning Rate: 0.04471910604647985
train_loss: 1.2785435184758371, val_loss: 1.474496637071882
epochs 4/84
Current Learning Rate: 0.04471910604647985
train_loss: 1.197447239416432, val_loss: 1.1755895999373582
epochs 5/84
Current Learning Rate: 0.04471910604647985
train_loss: 1.1269651832380845, val_loss: 1.0454602159520305
epochs 6/84
Current Learning Rate: 0.04471910604647985
train_loss: 1.0816962831307455, val_loss: 1.1379783632894043
epochs 7/84
Current Learning Rate: 0.04471910604647985
train_loss: 1.1054922223715258, val_loss: 1.1801792067825478
epochs 8/84
Current Learning Rate: 0.04471910604647985
train_loss: 1.0778300643591356, val_loss: 1.2143524412124875
epochs 9/84
Current Learning Rate: 0.04471910604647985
tra

[I 2024-01-27 01:51:26,578] Trial 0 finished with value: 1.1064366340404437 and parameters: {'batch_size': 86, 'epochs': 84, 'hidden_size': 51, 'learning_rate': 0.04471910604647985, 'dropout_prob': 0.11405534453545046, 'weight_decay': 8.740781938907536e-05, 'lr_step_size': 72, 'gamma': 0.19123176285413301, 'conv_channels': 84, 'kernel_size': 5, 'num_layers': 5, 'pool_size': 4, 'stride': 1}. Best is trial 0 with value: 1.1064366340404437.


Current Learning Rate: 0.04471910604647985
train_loss: 1.0364899819101174, val_loss: 1.2841643681601873
epochs 84/84
Current Learning Rate: 0.04471910604647985
train_loss: 1.035973895628324, val_loss: 1.2831430182885872
Mean validation loss: 1.1064366340404437
Starting fold 1:
epochs 1/129
Current Learning Rate: 4.1591821424223354e-05
train_loss: 1.6194562924469953, val_loss: 1.6391602111241175
epochs 2/129
Current Learning Rate: 4.1591821424223354e-05
train_loss: 1.6162133191892614, val_loss: 1.6370395718428192
epochs 3/129
Current Learning Rate: 4.1591821424223354e-05
train_loss: 1.6136898526346497, val_loss: 1.6349482826454929
epochs 4/129
Current Learning Rate: 4.1591821424223354e-05
train_loss: 1.6113840528807715, val_loss: 1.6328630781678295
epochs 5/129
Current Learning Rate: 4.1591821424223354e-05
train_loss: 1.6093816994372463, val_loss: 1.6307771445582153
epochs 6/129
Current Learning Rate: 4.1591821424223354e-05
train_loss: 1.6077674096791532, val_loss: 1.6286813910045321
ep

...

Current Learning Rate: 1.1286643068628048e-08
train_loss: 1.0358491208258753, val_loss: 1.2881981752536915
epochs 77/77
Current Learning Rate: 1.1286643068628048e-08
train_loss: 1.0320899059175817, val_loss: 1.2881981001959906
Mean validation loss: 1.1081558300073815
Starting fold 1:
epochs 1/89
Current Learning Rate: 0.09601613887467929
train_loss: 2.3586005184662904, val_loss: 6.558455618600997
epochs 2/89
Current Learning Rate: 0.09601613887467929
train_loss: 2.5104406060972764, val_loss: 1.3593076845956227
epochs 3/89
Current Learning Rate: 0.09601613887467929
train_loss: 1.2296425965443956, val_loss: 1.280878012142484
epochs 4/89
Current Learning Rate: 0.09601613887467929
train_loss: 1.47422551422219, val_loss: 1.2017664757985917
epochs 5/89
Current Learning Rate: 0.09601613887467929
train_loss: 1.1868331813063298, val_loss: 1.190112342910161
epochs 6/89
Current Learning Rate: 0.09601613887467929
train_loss: 1.1369965881577337, val_loss: 1.3421313403144715
epochs 7/89
Current Lear

[I 2024-01-27 01:55:11,417] Trial 23 pruned. 


Starting fold 1:
epochs 1/82
Current Learning Rate: 0.0007986509099890397
train_loss: 1.634462999423761, val_loss: 1.557348955875982
epochs 2/82
Current Learning Rate: 0.0007986509099890397
train_loss: 1.5662862249693945, val_loss: 1.4871567853544123
epochs 3/82
Current Learning Rate: 0.0007986509099890397
train_loss: 1.4758615368947934, val_loss: 1.3642313316385581
epochs 4/82
Current Learning Rate: 0.0007986509099890397
train_loss: 1.3036876336442238, val_loss: 1.2418738035928636
epochs 5/82
Current Learning Rate: 0.0007986509099890397
train_loss: 1.1658230108740442, val_loss: 1.2198205517713354
epochs 6/82
Current Learning Rate: 0.0007986509099890397
train_loss: 1.1192327176089063, val_loss: 1.1509541897546678
epochs 7/82
Current Learning Rate: 0.0007986509099890397
train_loss: 1.0733276018921618, val_loss: 1.0974090055183128
epochs 8/82
Current Learning Rate: 0.0007986509099890397
train_loss: 1.0793044910380978, val_loss: 1.0833981699413724
epochs 9/82
Current Learning Rate: 0.0007

[I 2024-01-27 01:55:16,271] Trial 24 pruned. 


Number of finished trials: 25
Best trial:
Value: 1.1064366340404437
Params:
batch_size: 86
epochs: 84
hidden_size: 51
learning_rate: 0.04471910604647985
dropout_prob: 0.11405534453545046
weight_decay: 8.740781938907536e-05
lr_step_size: 72
gamma: 0.19123176285413301
conv_channels: 84
kernel_size: 5
num_layers: 5
pool_size: 4
stride: 1


In [35]:
trainer = ModelActioner(train_data, test_data, device, model_type)
model = trainer.train(trial.params)

epochs 1/84
Current Learning Rate: 0.04471910604647985
train_loss: 1.2671875208616257
epochs 2/84
Current Learning Rate: 0.04471910604647985
train_loss: 1.1721922186478761
epochs 3/84
Current Learning Rate: 0.04471910604647985
train_loss: 1.0769790239946944
epochs 4/84
Current Learning Rate: 0.04471910604647985
train_loss: 1.092586436636851
epochs 5/84
Current Learning Rate: 0.04471910604647985
train_loss: 1.0823658549449813
epochs 6/84
Current Learning Rate: 0.04471910604647985
train_loss: 1.0773507148866923
epochs 7/84
Current Learning Rate: 0.04471910604647985
train_loss: 1.08044808966593
epochs 8/84
Current Learning Rate: 0.04471910604647985
train_loss: 1.0801034945417458
epochs 9/84
Current Learning Rate: 0.04471910604647985
train_loss: 1.0775346573389752
epochs 10/84
Current Learning Rate: 0.04471910604647985
train_loss: 1.0788547486906321
epochs 11/84
Current Learning Rate: 0.04471910604647985
train_loss: 1.078067997706608
epochs 12/84
Current Learning Rate: 0.04471910604647985


In [36]:
preds = trainer.test(trial.params)
y_true = y_test[time_step:]

accuracy = accuracy_score(y_true, preds)
print(f'Accuracy: {accuracy * 100:.2f}%')

# Precision
precision = precision_score(y_true, preds, average='weighted', zero_division=0)
print(f'Precision: {precision:.4f}')

# Recall
recall = recall_score(y_true, preds, average='weighted')
print(f'Recall: {recall:.4f}')

# F1 Score
f1 = f1_score(y_true, preds, average='weighted')
print(f'F1 Score: {f1:.4f}')

test_loss: 1.0864957072345385
Accuracy: 47.41%
Precision: 0.2248
Recall: 0.4741
F1 Score: 0.3050


In [37]:
unique_elements = set(preds)
print(unique_elements)

{0}


In [38]:
signals = pd.DataFrame(preds, columns=['Signal'])
signals

,Signal
0,0
1,0
2,0
3,0
4,0
...,...
246,0
247,0
248,0
249,0


In [39]:
signals["Signal"].value_counts()

Signal
0    251
Name: count, dtype: int64

In [40]:
signals["Date"] = date_test
signals

,Signal,Date
0,0,2023-01-23
1,0,2023-01-24
2,0,2023-01-25
3,0,2023-01-26
4,0,2023-01-27
...,...,...
246,0,2024-01-16
247,0,2024-01-17
248,0,2024-01-18
249,0,2024-01-19


In [41]:
stock_price = stock_data["Adj Close"].iloc[test_index:]
stock_price=stock_price.reset_index()
stock_price=stock_price.drop(columns=["index"])
stock_price

,Adj Close
0,140.325653
1,141.737762
2,141.071472
3,143.159821
4,145.118851
...,...
247,182.679993
248,188.630005
249,191.559998
250,193.889999


In [42]:
date_test["Date"] = date_test["Date"].dt.strftime('%Y-%m-%d')
date_test

,Date
0,2023-01-23
1,2023-01-24
2,2023-01-25
3,2023-01-26
4,2023-01-27
...,...
247,2024-01-17
248,2024-01-18
249,2024-01-19
250,2024-01-22


In [48]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=np.array(date_test).flatten(), y=stock_data["Adj Close"].iloc[test_index:], mode='lines', name='TSLA Stock Price'))

# Buy and sell signals
strong_sell_signals = signals[signals['Signal'] == 1]
sell_signals = signals[signals['Signal'] == 2]
buy_signals = signals[signals['Signal'] == 3]
strong_buy_signals = signals[signals['Signal'] == 4]


# Ensure that buy and sell prices are aligned with the signals by matching on the 'Date' column
buy_prices = stock_data[stock_data['Date'].isin(buy_signals['Date'])]["Adj Close"]
sell_prices = stock_data[stock_data['Date'].isin(sell_signals['Date'])]["Adj Close"]
# buy_prices = stock_data[stock_data['Date'].isin(buy_signals['Date'])]["Adj Close"]
# sell_prices = stock_data[stock_data['Date'].isin(sell_signals['Date'])]["Adj Close"]

# Plot buy signals
fig.add_trace(go.Scatter(
    x=buy_signals['Date'], 
    y=buy_prices, 
    mode='markers', 
    name='Buy', 
    marker=dict(color='green', size=10, symbol='triangle-up')
))

# Plot sell signals
fig.add_trace(go.Scatter(
    x=sell_signals['Date'], 
    y=sell_prices, 
    mode='markers', 
    name='Sell', 
    marker=dict(color='red', size=10, symbol='triangle-down')
))

# Update layout
fig.update_layout(
    title='Stock Price with Buy and Sell Signals',
    xaxis_title='Date',
    yaxis_title='Price',
    xaxis_rangeslider_visible=False,
    height = 700,
    width=1280
)

# Show the plot
pyo.iplot(fig)

/home/arda/anaconda3/lib/python3.11/site-packages/_plotly_utils/basevalidators.py:106: FutureWarning:

The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result



In [44]:
price_signal = stock_data[stock_data['Date'].isin(signals['Date'])]["Adj Close"]
price_signal = price_signal.reset_index()
price_signal = price_signal.drop(columns=["index"])
result = pd.concat([price_signal,signals], axis=1)
result

,Adj Close,Signal,Date
0,140.325653,0,2023-01-23
1,141.737762,0,2023-01-24
2,141.071472,0,2023-01-25
3,143.159821,0,2023-01-26
4,145.118851,0,2023-01-27
...,...,...,...
246,183.630005,0,2024-01-16
247,182.679993,0,2024-01-17
248,188.630005,0,2024-01-18
249,191.559998,0,2024-01-19


In [45]:
price_signal

,Adj Close
0,140.325653
1,141.737762
2,141.071472
3,143.159821
4,145.118851
...,...
246,183.630005
247,182.679993
248,188.630005
249,191.559998


In [46]:
sell_signals

,Signal,Date


In [47]:
buy_prices

Series([], Name: Adj Close, dtype: float64)